# Relational Algebra, SQL, and Integrity Constraints

**Syllabus mapping:** ER-model, relational model: relational algebra,
tuple calculus, SQL, integrity constraints, normal form.

**Objectives:** verify relational algebra operations (selection,
projection, natural join, set difference), translate relational
algebra to SQL, enforce integrity constraints, and test functional
dependencies.

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;")

# Schema with Primary Key and Foreign Key constraints
cursor.execute(
    """
    CREATE TABLE Students (
        sid INTEGER PRIMARY KEY,
        name TEXT NOT NULL,
        department TEXT NOT NULL
    );
    """
)

cursor.execute(
    """
    CREATE TABLE Enrollments (
        sid INTEGER,
        course_id TEXT,
        grade TEXT CHECK (grade IN ('A', 'B', 'C', 'F')),
        PRIMARY KEY (sid, course_id),
        FOREIGN KEY (sid) REFERENCES Students(sid) ON DELETE CASCADE
    );
    """
)

cursor.executemany(
    "INSERT INTO Students VALUES (?, ?, ?);",
    [
        (1, "Aarav", "CSE"),
        (2, "Diya", "AI"),
        (3, "Ishaan", "ECE"),
        (4, "Meera", "CSE"),
    ],
)

cursor.executemany(
    "INSERT INTO Enrollments VALUES (?, ?, ?);",
    [
        (1, "CS101", "A"),
        (1, "MA101", "B"),
        (2, "CS101", "A"),
        (2, "AI201", "A"),
        (3, "CS101", "B"),
    ],
)
conn.commit()
print("Database populated successfully.")

In [ ]:
# Relational Algebra 1: Selection (sigma_{department='CSE'}(Students))
print("--- Selection sigma_{department='CSE'} ---")
cursor.execute("SELECT * FROM Students WHERE department = 'CSE';")
print(cursor.fetchall())

# Relational Algebra 2: Projection (pi_{name}(Students))
print("\n--- Projection pi_{name} ---")
cursor.execute("SELECT name FROM Students;")
print(cursor.fetchall())

# Relational Algebra 3: Natural Join (Students bowtie Enrollments)
print("\n--- Natural Join (Students JOIN Enrollments) ---")
cursor.execute(
    """
    SELECT Students.sid, Students.name, Enrollments.course_id, Enrollments.grade
    FROM Students
    INNER JOIN Enrollments ON Students.sid = Enrollments.sid;
    """
)
for row in cursor.fetchall():
    print(row)

# Relational Algebra 4: Set Difference (pi_{sid}(Students) - pi_{sid}(Enrollments))
print("\n--- Students not enrolled in any course (Set Difference) ---")
cursor.execute(
    """
    SELECT sid, name FROM Students
    WHERE sid NOT IN (SELECT sid FROM Enrollments);
    """
)
print(cursor.fetchall())

## GATE-Style Practice

**MCQ:** In relational algebra, which algebraic expression represents
the relational division $R(A, B) \div S(B)$ (finding all $A$ values
associated with *every* $B$ value in $S$)?

A. $\pi_A(R) - \pi_A((\pi_A(R) \times S) - R)$
B. $\pi_A(R) \cap \pi_B(S)$
C. $\pi_A(R \bowtie S)$
D. $\pi_A(R) \cup S$

**MSQ:** Which of the following statements regarding database normalization
are true?

A. Every relational schema in BCNF is also in 3NF.
B. Any relational schema can be decomposed into a set of 3NF relations such that the decomposition is both lossless-join and dependency-preserving.
C. Any relational schema can always be decomposed into BCNF relations such that the decomposition is both lossless-join and dependency-preserving.
D. In BCNF, for every non-trivial functional dependency $X \to Y$, $X$ must be a superkey.

**NAT:** Given relation $R(A, B)$ with 6 tuples and relation $S(B, C)$
with 5 tuples. What is the **maximum possible** number of tuples in
the natural join $R \bowtie S$?

## Solutions

MCQ: **A**. By definition, relational division $R \div S$ produces the
tuples in $\pi_A(R)$ that do NOT fail to match some tuple in $S$. The
term $(\pi_A(R) \times S) - R$ gives all pairs $(a, b)$ missing from $R$;
projecting on $A$ and subtracting from $\pi_A(R)$ gives those $A$ values
present with all $B \in S$.

MSQ: **A, B, D**. (C is false: while a lossless-join BCNF decomposition
is always achievable, functional dependencies may not always be preserved
in BCNF).

NAT: **30**. If all 6 tuples of $R$ and all 5 tuples of $S$ share the
identical value for the join attribute $B$, every tuple of $R$ pairs with
every tuple of $S$, yielding $6 \times 5 = 30$ tuples.